# HELM — toxicité, cinq méthodes et quatre profils
Test fonctionnel sur deux commentaires de démonstration, modèle réel XLM-R toxicité. Cinq méthodes calculées : SHAP, LIME, IG natif avec contrôles numériques, Anchors natif et contrefactuels. Les erreurs et absences de résultat restent visibles. La sélection HELM par profil est distincte de ce test exhaustif ; les résultats complémentaires restent consultables.

## 1. Récupérer HELM depuis GitHub
Révision f717c7fb9ad14ab1e02e4863a72ea9ab0b83cc71. Dépôt privé : lien temporaire masqué. Les calculs sont exécutés dans Colab, sur GPU si disponible. Ce test ne constitue pas une validation sur un corpus représentatif ou auprès d’utilisateurs.


In [ ]:
import hashlib
import io
import sys
import urllib.request
import zipfile
from pathlib import Path
from getpass import getpass

REVISION = "f717c7fb9ad14ab1e02e4863a72ea9ab0b83cc71"
if globals().get("ARCHIVE_REVISION") != REVISION:
    archive_url = getpass("Lien GitHub temporaire de la révision : ")
    assert archive_url.startswith("https://codeload.github.com/KossiFO/helm-xai-these/")
    archive = urllib.request.urlopen(archive_url, timeout=120).read()
    del archive_url
    ARCHIVE_REVISION = REVISION
source_dir = Path("/content/helm_github_source")
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
    root_name = zipped.namelist()[0].split("/")[0]
    archive_revision = root_name.rsplit("-", 1)[-1]
    assert len(archive_revision) >= 7 and REVISION.startswith(archive_revision), "Révision inattendue"
    assert all((source_dir / n).resolve().is_relative_to(source_dir.resolve()) for n in zipped.namelist())
    zipped.extractall(source_dir)
SOURCE = source_dir / root_name
for module_name in list(sys.modules):
    if module_name == "helm_colab" or module_name.startswith("helm_colab."):
        del sys.modules[module_name]
sys.path.insert(0, str(SOURCE / "examples/colab"))
print("Codes récupérés depuis GitHub :", REVISION)
print("Archive SHA256 :", hashlib.sha256(archive).hexdigest())
print("Sources consultables dans le panneau Fichiers :", SOURCE)


## 2. Installer HELM dans Colab
Environnement Python 3.12 séparé, CPU ; le premier téléchargement du modèle demande quelques minutes.


In [ ]:
from helm_colab.toxicity_session import ToxicityExperiment
toxicite = ToxicityExperiment()
toxicite.install()


## 3. Calculer les cinq méthodes
Chaque méthode est exécutée une fois par commentaire puis réutilisée pour les quatre profils. IG utilise Captum et une référence PAD lexicale, sans LOO. Anchors utilise anchor-exp, sans remplacement par une recherche interne. Les limites numériques et les échecs sont conservés.


In [ ]:
toxicite.run()


## 4. Explorer les explications
Choisir le commentaire puis le profil. Le profil utilisateur explique le classement ; le modérateur reçoit les éléments à examiner ; l’expert compare les attributions ; l’audit donne la provenance.


In [ ]:
toxicite.show()


## 5. Détails techniques — facultatif
Cette section sert à reproduire et auditer le test : versions, modèle, paramètres, méthodes réussies ou en échec, et code réellement exécuté. Ouvrez le panneau seulement si nécessaire.


In [ ]:
toxicite.audit()
